# Notebook 0: Environment Check - GitHub Codespaces

**Python Applied to Business**

This notebook verifies the preinstalled course environment. It does **not** install libraries.

Run each cell in order. You only need to provide your personal `ZAI_API_KEY` when requested.


## Step 1: Verify the preinstalled environment

All dependencies are installed automatically when the Codespace is created.


In [ ]:
import sys
import importlib

checks = {
    "Python": None,
    "LangChain": "langchain",
    "LangChain OpenAI": "langchain_openai",
    "ChromaDB": "chromadb",
    "Sentence Transformers": "sentence_transformers",
    "LangChain HuggingFace": "langchain_huggingface",
    "CrewAI": "crewai",
    "CrewAI Tools": "crewai_tools",
    "PyPDF": "pypdf",
    "Unstructured": "unstructured",
    "Wikipedia": "wikipedia",
}

print(f"[OK] Python {sys.version.split()[0]}")
for label, module_name in checks.items():
    if module_name is None:
        continue
    module = importlib.import_module(module_name)
    version = getattr(module, "__version__", "installed")
    print(f"[OK] {label}: {version}")


## Step 2: Enter your GLM API key

The key is hidden while typing and is stored only in the current Python kernel session.


In [ ]:
import os
from getpass import getpass

if not os.getenv("ZAI_API_KEY"):
    os.environ["ZAI_API_KEY"] = getpass("Enter your ZAI_API_KEY: ").strip()

assert os.environ["ZAI_API_KEY"], "ZAI_API_KEY cannot be empty"
print("[OK] ZAI_API_KEY loaded for this notebook session")


## Step 3: Test the GLM connection through LangChain

The course uses the OpenAI-compatible endpoint exposed by Zhipu/BigModel.


In [ ]:
from langchain_openai import ChatOpenAI

GLM_MODEL = "glm-4.7-flash"
GLM_BASE_URL = "https://open.bigmodel.cn/api/paas/v4/"

def get_llm(temperature=0.3):
    return ChatOpenAI(
        model=GLM_MODEL,
        temperature=temperature,
        openai_api_key=os.environ["ZAI_API_KEY"],
        openai_api_base=GLM_BASE_URL,
    )

llm = get_llm()
response = llm.invoke("Reply only with: Hello, the connection works correctly.")
print("LLM replies:", response.content.strip())


## Step 4: Test local multilingual embeddings

Embeddings are local and do not consume LLM API quota. The model is preloaded during Codespace creation.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vector = embeddings.embed_query("This is an embeddings test for RAG")
print("[OK] Local embeddings work correctly")
print("Vector dimension:", len(vector))
print("First 5 values:", vector[:5])


## Step 5: Test ChromaDB using the same local embeddings

This avoids any hidden download of a second embedding model.


In [ ]:
import chromadb

client = chromadb.Client()
collection = client.create_collection("course_environment_test")

doc = "Python is a versatile programming language"
doc_embedding = embeddings.embed_query(doc)
collection.add(
    documents=[doc],
    embeddings=[doc_embedding],
    ids=["doc1"],
)

query_embedding = embeddings.embed_query("What is Python?")
results = collection.query(query_embeddings=[query_embedding], n_results=1)
print("[OK] ChromaDB works correctly")
print("Query result:", results["documents"][0][0])

client.delete_collection("course_environment_test")


## Step 6: Check the spring-course backup provider integrations

These are retained for compatibility with existing notebooks. No external API call is made here.


In [ ]:
available = []
for label, module_name, class_name in [
    ("Groq", "langchain_groq", "ChatGroq"),
    ("OpenAI", "langchain_openai", "ChatOpenAI"),
    ("Google Gemini", "langchain_google_genai", "ChatGoogleGenerativeAI"),
]:
    module = importlib.import_module(module_name)
    getattr(module, class_name)
    available.append(label)

print("[OK] Backup integrations installed:", ", ".join(available))


## Step 7: Test CrewAI with GLM

CrewAI uses the same OpenAI-compatible GLM endpoint. This test performs one minimal agent task.


In [ ]:
from crewai import Agent, Task, Crew, LLM

crew_llm = LLM(
    model=GLM_MODEL,
    custom_openai=True,
    base_url=GLM_BASE_URL,
    api_key=os.environ["ZAI_API_KEY"],
    max_tokens=256,
)

test_agent = Agent(
    role="Verifier",
    goal="Confirm that CrewAI works",
    backstory="You are a minimal environment test agent.",
    llm=crew_llm,
    verbose=False,
)

test_task = Task(
    description="Reply exactly: CrewAI works correctly.",
    expected_output="A short confirmation.",
    agent=test_agent,
)

crew = Crew(agents=[test_agent], tasks=[test_task], verbose=False)
result = crew.kickoff()
print("[OK] CrewAI executed successfully")
print("Result:", str(result)[:200])


## Verification summary

If every previous cell completed without errors, the Codespace is ready for the course.

| Component | Expected status |
|---|---|
| Preinstalled libraries | OK |
| Personal GLM API key | OK |
| GLM text generation | OK |
| Local multilingual embeddings | OK |
| ChromaDB | OK |
| Backup provider integrations | OK |
| CrewAI + GLM | OK |

If a test fails, stop and contact the instructor. Do not install or upgrade packages yourself.
